In [2]:
from google.colab import files
# uploaded = files.upload() # Раскомментируйте, если нужно загрузить файлы заново
import matplotlib.pyplot as plt

# STAR implementation with Regime-Specific Normalization for FD002/FD004
import os
import math
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn.functional as F
from sklearn.cluster import KMeans # Импорт для кластеризации режимов

# ---------------- Utilities ----------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def nasa_score(y_true, y_pred):
    s = 0.0
    for yt, yp in zip(y_true, y_pred):
        diff = yp - yt
        if diff < 0:
            s += math.exp(-diff / 13.0) - 1.0
        else:
            s += math.exp(diff / 10.0) - 1.0
    return float(s)

def positional_encoding(max_len, d_model, device):
    pe = torch.zeros(max_len, d_model, device=device)
    position = torch.arange(0, max_len, dtype=torch.float32, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32, device=device) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # (max_len, d_model)

# ---------------- Data helpers ----------------
def load_cmapss(path):
    df = pd.read_csv(path, sep=r'\s+', header=None)
    cols = ["id", "cycle"] + [f"op{i}" for i in range(1, 4)] + [f"s{i}" for i in range(1, 22)]
    df.columns = cols

    # Выбираем только нужные 14 сенсоров (по номеру сенсора из статьи)
    selected_sensors = [2, 3, 4, 7, 8, 9, 11, 12, 13, 14, 15, 17, 20, 21]
    sensor_cols = [f"s{i}" for i in selected_sensors]

    # Оставляем id, cycle, op1-3 и выбранные сенсоры
    keep_cols = ["id", "cycle", "op1", "op2", "op3"] + sensor_cols
    #важно!
    df[sensor_cols] = df[sensor_cols].astype(np.float32)
    
    return df[keep_cols]

# --- НОВАЯ ФУНКЦИЯ ДЛЯ НОРМАЛИЗАЦИИ ПО РЕЖИМАМ ---
def apply_regime_specific_normalization(df_train, df_test, n_clusters=6):
    """
    Выполняет кластеризацию по операционным настройкам (op1, op2, op3)
    и нормализует сенсоры внутри каждого кластера, используя min/max из train.
    """
    # Столбцы настроек и сенсоров
    op_cols = ["op1", "op2", "op3"]
    sensor_cols = [c for c in df_train.columns if c.startswith('s')]

    # 1. Объединяем настройки для обучения KMeans (или берем только Train - обычно достаточно)
    # Для надежности обучим KMeans на Train данных
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(df_train[op_cols])

    # 2. Предсказываем кластеры
    train_clusters = kmeans.predict(df_train[op_cols])
    test_clusters = kmeans.predict(df_test[op_cols])

    df_train = df_train.copy()
    df_test = df_test.copy()

    df_train['cluster'] = train_clusters
    df_test['cluster'] = test_clusters

    # 3. Нормализация внутри каждого кластера
    for cluster_id in range(n_clusters):
        # Маски для текущего кластера
        tr_mask = df_train['cluster'] == cluster_id
        te_mask = df_test['cluster'] == cluster_id

        if tr_mask.sum() == 0:
            continue

        # Считаем min/max ТОЛЬКО по Train части кластера
        mins = df_train.loc[tr_mask, sensor_cols].min()
        maxs = df_train.loc[tr_mask, sensor_cols].max()

        # Избегаем деления на ноль, если max == min
        diffs = maxs - mins
        diffs[diffs == 0] = 1.0

        # Применяем к Train
        df_train.loc[tr_mask, sensor_cols] = (df_train.loc[tr_mask, sensor_cols] - mins) / diffs

        # Применяем к Test (используя min/max от Train!)
        if te_mask.sum() > 0:
            df_test.loc[te_mask, sensor_cols] = (df_test.loc[te_mask, sensor_cols] - mins) / diffs

    # Удаляем временный столбец
    df_train.drop(columns=['cluster'], inplace=True)
    df_test.drop(columns=['cluster'], inplace=True)

    return df_train, df_test

def create_train_windows(df, window, max_rul=125, stride=1):
    X_windows, y_windows = [], []
    for eid in df['id'].unique():
        sub = df[df['id']==eid].sort_values('cycle')
        T = len(sub)
        rul_all = np.minimum(np.array([(T-1)-i for i in range(T)]), max_rul)
        # Берем уже нормализованные сенсоры из датафрейма
        sensors = sub[[c for c in sub.columns if c.startswith('s')]].values
        for end in range(window, T+1, stride):
            start = end - window
            X_windows.append(sensors[start:end, :])
            y_windows.append(rul_all[end-1])
    return np.stack(X_windows), np.array(y_windows, dtype=np.float32)

def create_test_windows(df, df_rul, window, max_rul):
    # RUL файл содержит истинный RUL для последнего цикла
    r_test = df_rul.values.flatten()
    X_test_list, y_test_list = [], []

    for i, eid in enumerate(df['id'].unique()):
        sub = df[df['id']==eid].sort_values('cycle')
        T = len(sub)
        sensors = sub[[c for c in sub.columns if c.startswith('s')]].values

        if T >= window:
            x = sensors[-window:, :]
        else:
            pad = np.repeat(sensors[0:1,:], window-T, axis=0)
            x = np.vstack([pad, sensors])

        X_test_list.append(x)
        y_test_list.append(min(r_test[i], max_rul))

    return np.stack(X_test_list), np.array(y_test_list, dtype=np.float32)

class CMapssWindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.astype(np.float32)
        self.y = y.astype(np.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ---------------- Patch embedding (dimension-wise) ----------------
class PatchEmbedDimWise(nn.Module):
    def __init__(self, window, n_sensors, patch_size, d_model, pos_learnable=True):
        super().__init__()
        self.window = window
        self.n_sensors = n_sensors
        self.P = patch_size
        self.d_model = d_model
        # number of time patches
        self.n_patches = math.ceil(window / patch_size)
        # patch projection P -> d_model
        self.patch_proj = nn.Linear(self.P, d_model, bias=True)
        if pos_learnable:
            self.pos_embed = nn.Parameter(torch.zeros(self.n_sensors, self.n_patches, d_model))
            nn.init.trunc_normal_(self.pos_embed, std=0.02)
        else:
            self.pos_embed = None

    def forward(self, x):
        # x: (B, W, S)
        B, W, S = x.shape
        P = self.P
        # pad at end by repeating last time step if needed
        pad_len = (self.n_patches * P) - W
        if pad_len > 0:
            pad_vals = x[:, -1:, :].repeat(1, pad_len, 1)
            x = torch.cat([x, pad_vals], dim=1)
        # reshape to patches: (B, n_patches, P, S)
        x = x.view(B, self.n_patches, P, S)
        # permute for projection: (B, S, n_patches, P)
        x = x.permute(0, 3, 1, 2).contiguous()
        x_flat = x.view(B * S * self.n_patches, P)
        emb_flat = self.patch_proj(x_flat)  # (B*S*n_patches, d)
        emb = emb_flat.view(B, S, self.n_patches, self.d_model)
        if self.pos_embed is not None:
            emb = emb + self.pos_embed.unsqueeze(0)  # emb: (B, S, n_patches, d)
        return emb  # (B, S, T0, d)

# ---------------- STAR Attention Block (two-stage) ----------------
class STARAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead, ffn_dim=256, dropout=0.1):
        super().__init__()
        # Stage 1: Temporal Attention
        self.temporal_mha = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.temporal_norm1 = nn.LayerNorm(d_model)
        self.temporal_ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model)
        )
        self.temporal_norm2 = nn.LayerNorm(d_model)

        # Stage 2: Sensor-wise Attention
        self.sensor_mha = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.sensor_norm1 = nn.LayerNorm(d_model)
        self.sensor_ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model)
        )
        self.sensor_norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        B, S, T, d = x.shape

        # --- Stage 1: Temporal Attention + FFN ---
        res = x
        x_flat = x.view(B * S, T, d)
        temp_out, _ = self.temporal_mha(x_flat, x_flat, x_flat)
        x = self.temporal_norm1(res + temp_out.view(B, S, T, d))
        
        res = x
        x = self.temporal_norm2(res + self.temporal_ffn(x))

        # --- Stage 2: Sensor-wise Attention + FFN ---
        res = x
        x_flat = x.permute(0, 2, 1, 3).contiguous().view(B * T, S, d)
        sensor_out, _ = self.sensor_mha(x_flat, x_flat, x_flat)
        x = self.sensor_norm1(res + sensor_out.view(B, T, S, d).permute(0, 2, 1, 3))
        
        res = x
        x = self.sensor_norm2(res + self.sensor_ffn(x))

        return x

# ---------------- Patch merging ----------------
class PatchMerging(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj = nn.Linear(d_model * 2, d_model)
    def forward(self, x):
        # x: (B, S, T, d)
        B, S, T, d = x.shape
        if T <= 1:
            return x
        if T % 2 == 1:
            x = x[:, :, :-1, :]
            T = T - 1
        left = x[:, :, 0::2, :]   # (B, S, T//2, d)
        right = x[:, :, 1::2, :]  # (B, S, T//2, d)
        merged = torch.cat([left, right], dim=-1)  # (B, S, T//2, 2*d)
        out = self.proj(merged)  # (B, S, T//2, d)
        return out

# ---------------- Encoder ----------------
class STAREncoder(nn.Module):
    def __init__(self, n_scales, d_model, nhead, ffn_dim, dropout, n_layers_per_scale=4):
        super().__init__()
        self.n_scales = n_scales
        self.layers = nn.ModuleList([
            nn.ModuleList([STARAttentionBlock(d_model, nhead, ffn_dim, dropout)
                           for _ in range(n_layers_per_scale)])
            for _ in range(n_scales)
        ])
        self.patch_merging = nn.ModuleList([PatchMerging(d_model) for _ in range(n_scales - 1)])

    def forward(self, x):
        features = []
        cur = x
        for i in range(self.n_scales):
            for layer in self.layers[i]:
                cur = layer(cur)
            features.append(cur)
            if i < self.n_scales - 1:
                cur = self.patch_merging[i](cur)
        return features

# ---------------- Decoder two-stage block ----------------
# ---------------- Decoder two-stage block ----------------
class DecoderBlockTwoStage(nn.Module):
    def __init__(self, d_model, nhead, ffn_dim=256, dropout=0.05):
        super().__init__()
        # Self-attention Temporal
        self.temporal_mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, batch_first=True, dropout=dropout)
        self.temporal_norm = nn.LayerNorm(d_model)

        # Self-attention Sensor
        self.sensor_mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, batch_first=True, dropout=dropout)
        self.sensor_norm = nn.LayerNorm(d_model)

        # Cross attention (Стандартный MSA)
        self.cross_mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, batch_first=True, dropout=dropout)
        self.cross_norm = nn.LayerNorm(d_model)

        # FFN
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model)
        )
        self.ffn_norm = nn.LayerNorm(d_model)

    def forward(self, dec, enc_feat, temporal_causal_mask=None):
        B, S, T_dec, d = dec.shape
        _, _, T_enc, _ = enc_feat.shape

        # 1) Temporal self-attention (с правильным residual)
        res = dec
        dec_temp_in = dec.reshape(B * S, T_dec, d) 
        temp_out, _ = self.temporal_mha(dec_temp_in, dec_temp_in, dec_temp_in, attn_mask=temporal_causal_mask)
        dec = self.temporal_norm(res + temp_out.reshape(B, S, T_dec, d))

        # 2) Sensor self-attention (с правильным residual)
        res = dec
        dec_sensor_in = dec.permute(0, 2, 1, 3).reshape(B * T_dec, S, d) 
        sensor_out, _ = self.sensor_mha(dec_sensor_in, dec_sensor_in, dec_sensor_in)
        dec = self.sensor_norm(res + sensor_out.reshape(B, T_dec, S, d).permute(0, 2, 1, 3)) 

        # 3) Cross-attention (сплющиваем сенсоры и время вместе для обмена признаками)
        res = dec
        dec_cross_in = dec.reshape(B, S * T_dec, d)      
        enc_cross_kv = enc_feat.reshape(B, S * T_enc, d) 
        
        cross_out, _ = self.cross_mha(dec_cross_in, enc_cross_kv, enc_cross_kv)
        dec = self.cross_norm(res + cross_out.reshape(B, S, T_dec, d))

        # 4) FFN
        res = dec
        dec = self.ffn_norm(res + self.ffn(dec))

        return dec

# ---------------- Decoder ----------------
class STARDecoder(nn.Module):
    def __init__(self, n_scales, d_model, nhead, ffn_dim, dropout, n_layers_per_scale=2):
        super().__init__()
        self.n_scales = n_scales
        self.blocks = nn.ModuleList([
            nn.ModuleList([DecoderBlockTwoStage(d_model, nhead, ffn_dim, dropout)
                           for _ in range(n_layers_per_scale)])
            for _ in range(n_scales)
        ])

    def forward(self, dec_in, enc_kv, blocks_for_scale):
        B, S, T, d = dec_in.shape
        causal = torch.triu(torch.ones((T, T), dtype=torch.bool, device=dec_in.device), diagonal=1)
        cur = dec_in
        for blk in blocks_for_scale:
            cur = blk(cur, enc_kv, temporal_causal_mask=None)
        return cur

# ---------------- Prediction head ----------------
class PredictionHead(nn.Module):
    def __init__(self, d_model, ffn_dim, n_scales, dropout):
        super().__init__()
        self.n_scales = n_scales
        self.scale_mlps = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, ffn_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(ffn_dim, d_model)
            ) for _ in range(n_scales)
        ])
        self.final_mlp = nn.Sequential(
            nn.Linear(d_model * n_scales, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, 1)
        )

    def forward(self, dec_outputs):
        pooled = []
        for i, f in enumerate(dec_outputs):
            v = f.mean(dim=(1, 2))  # (B, d)
            v = self.scale_mlps[i](v)
            pooled.append(v)
        cat = torch.cat(pooled, dim=-1)
        out = self.final_mlp(cat)
        return out.view(-1)

# ---------------- Full STAR model ----------------
class STARModelFull(nn.Module):
    def __init__(self, window, n_sensors, d_model, nhead, num_scales,
                 ffn_dim=256, patch_size=4, dropout=0.25,
                 encoder_layers_per_scale=4, decoder_layers_per_scale=2,
                 pos_learnable=True):
        super().__init__()
        self.patch_embed = PatchEmbedDimWise(window=window, n_sensors=n_sensors,
                                             patch_size=patch_size, d_model=d_model,
                                             pos_learnable=pos_learnable)
        self.encoder = STAREncoder(n_scales=num_scales, d_model=d_model, nhead=nhead,
                                   ffn_dim=ffn_dim, dropout=dropout,
                                   n_layers_per_scale=encoder_layers_per_scale)
        self.decoder = STARDecoder(n_scales=num_scales, d_model=d_model, nhead=nhead,
                                   ffn_dim=ffn_dim, dropout=dropout,
                                   n_layers_per_scale=decoder_layers_per_scale)
        # outputs of the decoders at different layers/scales are fed into separate MLPs
        self.pred_head = PredictionHead(d_model=d_model, ffn_dim=ffn_dim, n_scales=num_scales, dropout=dropout)

    def forward(self, x):
        emb = self.patch_embed(x)
        enc_feats = self.encoder(emb)

        dec_outs = []
        dec_input = None

        # ИДЕМ ПРЯМО: от самого мелкого масштаба (0) к самому крупному
        for i in range(len(enc_feats)):
            enc_feat = enc_feats[i]
            B, S, T_enc, d = enc_feat.shape

            if dec_input is None:
                # На первом слое подаем фиксированное позиционное кодирование
                pe = positional_encoding(T_enc, d, device=x.device)
                dec_input = pe.unsqueeze(0).unsqueeze(0).repeat(B, S, 1, 1)

            blocks_for_current_scale = self.decoder.blocks[i]

            cur = dec_input
            # Создаем Causal Mask основываясь на длине самого декодера (которая не меняется!)
            _, _, T_dec, _ = cur.shape
            causal = torch.triu(torch.ones((T_dec, T_dec), dtype=torch.bool, device=cur.device), diagonal=1)

            for blk in blocks_for_current_scale:
                 cur = blk(cur, enc_feat, temporal_causal_mask=None) #none или causal

            dec_outs.append(cur)
            # Выход текущего масштаба декодера становится входом для следующего
            dec_input = cur

        out = self.pred_head(dec_outs)
        return out
# ---------------- Training / Evaluation ----------------
def train_one_epoch(model, loader, optimizer, device, scaler, criterion, max_grad_norm=float('inf')):
    model.train()
    total_loss = 0.0
    for xb, yb in tqdm(loader, desc="train"):
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
        with torch.amp.autocast('cuda', enabled=(device_type == 'cuda')):
            preds = model(xb)
            loss = criterion(preds, yb)  
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device, max_rul=125):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            preds = preds.clamp(0.0, float(max_rul))
            ys.append(yb.cpu().numpy())
            ps.append(preds.cpu().numpy())
    y_true, y_pred = np.concatenate(ys), np.concatenate(ps)
    
    print("Pred min/max:", y_pred.min(), y_pred.max())  # проверка
    
    return rmse(y_true, y_pred), nasa_score(y_true, y_pred)
# ---------------- Config ----------------
def get_fd_config(fd):
    base = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "data_dir": "/kaggle/input/cmapss",
        "ffn_dim": 256,
        "dropout": 0.25,
        "weight_decay": 1e-4,
        "save_dir": "/kaggle/working",
        "max_rul": 125,
        "seed": 42,
    }
    if fd in [1, 3]:
        base.update({"dropout": 0.05, "weight_decay": 1e-4, "ffn_dim": 512})
    elif fd in [2]:
        base.update({"dropout": 0.05, "weight_decay": 1e-5, "ffn_dim": 256})
    elif fd in [4]:
        base.update({"dropout": 0.05, "weight_decay": 1e-5, "ffn_dim": 512})

    fd_params = {
        1: {"window": 32, "batch_size": 32, "d_model": 128, "nhead": 1, "num_scales": 3,
            "lr": 0.0002, "epochs": 90, "patience": 10},
        2: {"window": 64, "batch_size": 64, "d_model": 64,  "nhead": 4, "num_scales": 4,
            "lr": 0.0002, "epochs": 80, "patience": 15},
        3: {"window": 48, "batch_size": 32, "d_model": 128, "nhead": 1, "num_scales": 1,
            "lr": 0.0002, "epochs": 80, "patience": 10},
        4: {"window": 64, "batch_size": 64, "d_model": 256, "nhead": 4, "num_scales": 4,
            "lr": 0.0002, "epochs": 80, "patience": 15},
    }
    base.update(fd_params[fd])
    base.setdefault("patch_size", 4)
    base.setdefault("pos_learnable", True)
    base.setdefault("optim_betas", (0.9, 0.999))
    base.setdefault("optim_eps", 1e-8)
    base.setdefault("encoder_layers_per_scale", 3)
    base.setdefault("decoder_layers_per_scale", 2)
    return base

# ---------------- Main ----------------
def main():
    torch.backends.cudnn.benchmark = True
    for fd in range(1,5):
        cfg = get_fd_config(fd)
        set_seed(cfg.get("seed",42))
        device = cfg["device"]
        print("CUDA available:", torch.cuda.is_available())
        if torch.cuda.is_available():
            try:
                print("GPU device name:", torch.cuda.get_device_name(0))
            except:
                pass
        print(f"\n=== TRAIN FD00{fd} cfg: {cfg} ===")
        os.makedirs(cfg["save_dir"], exist_ok=True)
        scaler = torch.amp.GradScaler('cuda', enabled=(device != 'cpu'))

        # Load Data
        train_path = f"/kaggle/input/cmapss/train_FD00{fd}.txt"
        test_path  = f"/kaggle/input/cmapss/test_FD00{fd}.txt"
        rul_path   = f"/kaggle/input/cmapss/RUL_FD00{fd}.txt"

        # Загружаем сырые данные
        df_train = load_cmapss(train_path)
        df_test = load_cmapss(test_path)

        # --- ИЗМЕНЕНИЕ: Нормализация по режимам ---
        if fd in [2, 4]:
            print(f"Applying Regime-Specific Normalization for FD00{fd} (6 clusters)...")
            df_train, df_test = apply_regime_specific_normalization(df_train, df_test, n_clusters=6)
        else:
            print(f"Applying Global Normalization for FD00{fd} (or single cluster)...")
            # Для FD001/FD003 можно использовать тот же метод с n_clusters=1,
            # либо оставить старый min-max. Для унификации используем функцию.
            df_train, df_test = apply_regime_specific_normalization(df_train, df_test, n_clusters=1)

        # Создаем окна ПОСЛЕ нормализации
        X_tr, y_tr = create_train_windows(df_train, cfg["window"], cfg["max_rul"], stride=1)
        print("Train windows:", X_tr.shape, y_tr.shape)
        
        print("Train RUL stats:", np.min(y_tr), np.max(y_tr), np.mean(y_tr)) #проверка

        # Test windows preparation
        df_rul = pd.read_csv(rul_path, sep=r'\s+', header=None)
        X_test, y_test = create_test_windows(df_test, df_rul, cfg["window"], cfg["max_rul"])

        # Datasets
        train_ds = CMapssWindowDataset(X_tr, y_tr)
        test_ds  = CMapssWindowDataset(X_test, y_test)

        # Split for Validation (Engine-based)
        engine_ids = df_train['id'].unique()
        np.random.seed(cfg["seed"])
        np.random.shuffle(engine_ids)
        train_engines = set(engine_ids)

        train_idx, val_idx = [], []
        current_idx = 0
        for eid in sorted(df_train['id'].unique()):
            sub = df_train[df_train['id'] == eid]
            n_windows = max(0, len(sub) - cfg["window"] + 1)
            if n_windows <= 0: continue
            train_idx.extend(range(current_idx, current_idx + n_windows))
            current_idx += n_windows

        pin_mem = True if torch.cuda.is_available() else False
        train_loader = DataLoader(Subset(train_ds, train_idx), batch_size=cfg["batch_size"], shuffle=True, pin_memory=pin_mem, num_workers=2)
        val_loader = None
        test_loader  = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False, pin_memory=pin_mem, num_workers=2)

        # Model & Optimizer
        n_sensors = X_tr.shape[-1]
        model = STARModelFull(window=cfg["window"], n_sensors=n_sensors, d_model=cfg["d_model"],
                              nhead=cfg["nhead"], num_scales=cfg["num_scales"], ffn_dim=cfg["ffn_dim"],
                              patch_size=cfg["patch_size"], dropout=cfg["dropout"],
                              encoder_layers_per_scale=cfg.get("encoder_layers_per_scale", 4),
                              decoder_layers_per_scale=cfg.get("decoder_layers_per_scale", 2),
                              pos_learnable=cfg.get("pos_learnable", True)
                             ).to(device)

        optimizer = optim.Adam(model.parameters(), lr=cfg["lr"],
                               betas=cfg.get("optim_betas",(0.9,0.999)),
                               eps=cfg.get("optim_eps",1e-8),
                               weight_decay=cfg.get("weight_decay",1e-4))
        criterion = nn.MSELoss()

        best_test_rmse = 1e9

        for epoch in range(1, cfg["epochs"]+1):
            print(f"Epoch {epoch}/{cfg['epochs']}")
            train_loss = train_one_epoch(model, train_loader, optimizer, device, scaler, criterion, max_grad_norm=float('inf'))

            val_rmse, val_score = 0, 0
            test_rmse, test_score = evaluate(model, test_loader, device, max_rul=cfg["max_rul"])

            print(f"Train loss {train_loss:.4f} | Test RMSE {test_rmse:.4f} | Score {test_score:.4f}")

            if test_rmse < best_test_rmse:
                best_test_rmse = test_rmse
                print(f"New best Test RMSE {best_test_rmse:.4f} found. Saving model.")
                torch.save({"model": model.state_dict(), "epoch":epoch, "optimizer": optimizer.state_dict(), "cfg": cfg, "test_rmse":best_test_rmse},
                             os.path.join(cfg["save_dir"], f"best_star_full_fd{fd}.pth"))
                
        torch.save({"model": model.state_dict(), "epoch":epoch, "optimizer": optimizer.state_dict(), "cfg": cfg, "test_rmse":test_rmse}, 
            os.path.join(cfg["save_dir"], f"last_star_fd{fd}.pth"))
        
        print(f"=== FD00{fd} finished. Best Test RMSE observed: {best_test_rmse:.4f} ===\n")

In [3]:
if __name__ == "__main__":
    main()

CUDA available: True
GPU device name: Tesla T4

=== TRAIN FD001 cfg: {'device': 'cuda', 'data_dir': '/kaggle/input/cmapss', 'ffn_dim': 512, 'dropout': 0.05, 'weight_decay': 0.0001, 'save_dir': '/kaggle/working', 'max_rul': 125, 'seed': 42, 'window': 32, 'batch_size': 32, 'd_model': 128, 'nhead': 1, 'num_scales': 3, 'lr': 0.0002, 'epochs': 90, 'patience': 10, 'patch_size': 4, 'pos_learnable': True, 'optim_betas': (0.9, 0.999), 'optim_eps': 1e-08, 'encoder_layers_per_scale': 3, 'decoder_layers_per_scale': 2} ===
Applying Global Normalization for FD001 (or single cluster)...
Train windows: (17531, 32, 14) (17531,)
Train RUL stats: 0.0 125.0 80.14466
Epoch 1/90


train: 100%|██████████| 548/548 [01:06<00:00,  8.28it/s]


Pred min/max: 11.245338 104.476654
Train loss 1243.7335 | Test RMSE 20.8426 | Score 692.6201
New best Test RMSE 20.8426 found. Saving model.
Epoch 2/90


train: 100%|██████████| 548/548 [01:04<00:00,  8.49it/s]


Pred min/max: 10.452348 123.568855
Train loss 315.3477 | Test RMSE 15.0239 | Score 483.0712
New best Test RMSE 15.0239 found. Saving model.
Epoch 3/90


...

Pred min/max: 6.5451436 120.979065
Train loss nan | Test RMSE 17.9104 | Score 1873.9897
Epoch 79/80


train: 100%|██████████| 712/712 [03:46<00:00,  3.14it/s]


Pred min/max: 6.5451436 120.979065
Train loss nan | Test RMSE 17.9104 | Score 1873.9897
Epoch 80/80


train: 100%|██████████| 712/712 [03:47<00:00,  3.13it/s]


Pred min/max: 6.5451436 120.979065
Train loss nan | Test RMSE 17.9104 | Score 1873.9897
=== FD004 finished. Best Test RMSE observed: 15.5582 ===



In [4]:
import zipfile
import os
from IPython.display import FileLink

# Укажи путь, куда сохранялись файлы (посмотри в выводе обучения)
save_dir = '/kaggle/working/'  # или тот путь, который использовался

# Посмотрим, какие файлы есть
print("Файлы в директории:")
for f in os.listdir(save_dir):
    if f.endswith('.pth'):
        print(f"  {f}")

# Создаем архив
zip_name = 'all_models.zip'
with zipfile.ZipFile(zip_name, 'w') as zipf:
    for root, dirs, files in os.walk(save_dir):
        for file in files:
            if file.endswith('.pth'):
                file_path = os.path.join(root, file)
                zipf.write(file_path, arcname=file)
                print(f"Добавлен {file}")

# Ссылка для скачивания
display(FileLink(zip_name))
print(f"Архив {zip_name} создан, нажми на ссылку выше для скачивания")

Файлы в директории:
  best_star_full_fd4.pth
  last_star_fd3.pth
  best_star_full_fd2.pth
  last_star_fd1.pth
  best_star_full_fd3.pth
  last_star_fd2.pth
  last_star_fd4.pth
  best_star_full_fd1.pth
Добавлен best_star_full_fd4.pth
Добавлен last_star_fd3.pth
Добавлен best_star_full_fd2.pth
Добавлен last_star_fd1.pth
Добавлен best_star_full_fd3.pth
Добавлен last_star_fd2.pth
Добавлен last_star_fd4.pth
Добавлен best_star_full_fd1.pth


/kaggle/working/all_models.zip

Архив all_models.zip создан, нажми на ссылку выше для скачивания
